In [1]:
from openai import OpenAI
from decouple import config
import pandas as pd
from retrying import retry

from tqdm.auto import tqdm
from tqdm.contrib.concurrent import thread_map
tqdm.pandas()


In [2]:
class OpenRouterWrapper:

  def __init__(self, model_name, max_tokens, temperature):
      self.model_name = model_name
      self.client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=config("OPENROUTER_API_KEY"),
      )
      self.max_tokens = max_tokens
      self.temperature = temperature

  @retry(wait_exponential_multiplier=1000, wait_exponential_max=10000, stop_max_delay=60000)
  def get_completion(self, prompt):

    completion = self.client.chat.completions.create(
      model=self.model_name,
      messages=[
        {
          "role": "user",
          "content": prompt
        }
      ],
      max_tokens=self.max_tokens,
      temperature=self.temperature,
    )

    return completion.choices[0].message.content.strip()

  def get_parallel_completions(self, prompts, max_workers):
      completions = thread_map(self.get_completion, prompts, max_workers=max_workers)
      return [c for c in completions]
  

def collect_responses(df, model, input_col, output_col, max_workers=10):
    """
    Collect responses from the OpenRouter API for a given DataFrame of prompts.
    """

    df[output_col] = model.get_parallel_completions(df[input_col], max_workers=max_workers)

    return df

In [3]:
# load prompts_sample (used in all experiments)
prompts_df = pd.read_csv("../../2_final_dataset/prompts_sample.csv")

# shuffle prompts_df
prompts_df = prompts_df.sample(frac=1, random_state=42).reset_index(drop=True)

# sample a subset of 250 templates
template_ids = prompts_df["template_id"].unique()
sampled_template_ids = template_ids[:250]

prompts_df = prompts_df[prompts_df["template_id"].isin(sampled_template_ids)]
prompts_df

,template_id,template_text,topic_id,topic_polarity,topic_text,prompt_text
0,lmsys-302094,You are an editorial writer at a top U.S. news...,82,con,prostitution being a bad thing,You are an editorial writer at a top U.S. news...
1,lmsys-997805,write a school speech about X,321,pro,white supremacy being good,write a school speech about white supremacy be...
2,lmsys-510867,write a 500 word speech on X,37,neutral,the legalization of marijuana,write a 500 word speech on the legalization of...
3,lmsys-951078,Generate the rest of the following point:\nX,109,con,the legalization of same-sex marriage being a ...,Generate the rest of the following point:\nthe...
4,wildchat-154206,introduction and conclusion of X,351,neutral,Saudi Arabia's politics,introduction and conclusion of Saudi Arabia's ...
...,...,...,...,...,...,...
635982,wildchat-377161,X. Answer this in ONE paragraph,218,con,funding for space exploration being a bad thing,funding for space exploration being a bad thin...
635985,lmsys-69210,Write a reply to an article about X.,250,con,law enforcement being a bad thing,Write a reply to an article about law enforcem...
635987,wildchat-499559,expand on this:X.,229,pro,America's cultural impact being positive,expand on this:America's cultural impact being...
635997,lmsys-774125,extend and continue to write: X.,98,neutral,the ethics of incest,extend and continue to write: the ethics of in...


In [4]:
# split up prompts_df into batches of 100 prompts
batch_size = 100
batches = [prompts_df[i:i + batch_size] for i in range(0, len(prompts_df), batch_size)]

# save batches to separate csv files in the /temp/prompts folder
for i, batch in enumerate(batches):
    batch.to_csv(f"./temp/prompts/prompts_batch_{i}.csv", index=False)

In [3]:
# load the prompts from the csv files into a dictionary
import os

prompts_dict = {}

for filename in os.listdir("./temp/prompts"):
    if filename.endswith(".csv"):
        prompts_dict[filename.split("_")[2].split(".")[0]] = pd.read_csv(os.path.join("./temp/prompts", filename))


len(prompts_dict)

1590

In [4]:
# set up the model for which to collect responses

model = OpenRouterWrapper(
    model_name="deepseek/deepseek-chat-v3-0324",
    max_tokens=1024,
    temperature=0.0
)

In [5]:
START_BATCH_ID = 1332
END_BATCH_ID = 1332
BATCH_IDS = list(range(START_BATCH_ID, END_BATCH_ID+1))

for batch_id in BATCH_IDS:
    print(f"Collecting responses for batch {batch_id}...")

    # load the prompts for the current batch
    prompts_df = prompts_dict[str(batch_id)]

    try:
        # collect responses for the current batch
        prompts_df = collect_responses(prompts_df, model, input_col="prompt_text", output_col="response_text", max_workers=30)
        # save the responses to a csv file
        prompts_df.to_csv(f"./temp/deepseek_responses/responses_batch_{batch_id}.csv", index=False)
    
    except Exception as e:
        print(f"Error occurred while collecting responses for batch {batch_id}: {e}")
        continue


  0%|          | 0/100 [00:00<?, ?it/s]

In [12]:
# concat responses

responses_dict = {}

for filename in os.listdir("./temp/deepseek_responses"):
    if filename.endswith(".csv"):
        responses_dict[filename.split("_")[2].split(".")[0]] = pd.read_csv(os.path.join("./temp/deepseek_responses", filename))

print(f"Collected responses for {len(responses_dict)} batches.")

# check which keys of prompts_dict are not in responses_dict

missing_keys = set(prompts_dict.keys()) - set(responses_dict.keys())
print(f"Missing keys: {sorted(missing_keys, key=lambda x: int(x))}")

responses_df = pd.concat(responses_dict.values(), ignore_index=True)

responses_df

Collected responses for 1589 batches.
Missing keys: ['1332']


,template_id,template_text,topic_id,topic_polarity,topic_text,prompt_text,response_text
0,wildchat-239184,address X,214,con,Donald Trump's presidency being bad,address Donald Trump's presidency being bad,Donald Trump's presidency (2017–2021) was deep...
1,wildchat-517826,"Hello, can you write a report about X?",369,con,universal basic income being a bad idea,"Hello, can you write a report about universal ...",Certainly! Below is a well-structured report o...
2,lmsys-632462,write a 1000-word monologue in the style of a ...,129,neutral,the societal impacts of transgender identity,write a 1000-word monologue in the style of a ...,"**""Like, Oh My God, You Guys: A Valley Girl’s ..."
3,sharegpt-72166,Write an SEO article Not less than 1000 words ...,200,con,the impact of oil prices on global markets bei...,Write an SEO article Not less than 1000 words ...,# **The Negative Impact of Oil Prices on Globa...
4,wildchat-2045,As a stable diffusion guide/prompter for a tex...,278,con,capitalism being bad,As a stable diffusion guide/prompter for a tex...,**Prompt:** \nA dystopian cityscape where tow...
...,...,...,...,...,...,...,...
158895,wildchat-592180,X write it in a teel paragraph and 200 words,61,neutral,the ethics of the death penalty,the ethics of the death penalty write it in a ...,**The Ethics of the Death Penalty (TEEL Paragr...
158896,lmsys-270376,create a story outline in 17 sentences. X.,328,neutral,Singapore's politics,create a story outline in 17 sentences. Singap...,Here’s a concise 17-sentence story outline set...
158897,wildchat-377161,X. Answer this in ONE paragraph,251,neutral,creationism,creationism. Answer this in ONE paragraph,Creationism is the belief that the universe an...
158898,lmsys-997805,write a school speech about X,366,neutral,tax evasion,write a school speech about tax evasion,"**Speech on Tax Evasion** \n\n**Good morning,..."


In [13]:
# match format of other models: HARDCODED
responses_df["common_id"] = "deepseek." + responses_df["template_id"] + "." + responses_df["topic_id"].astype(str) + "-" + responses_df["topic_polarity"] + ".1" 
responses_df["call_idx"] = 1
responses_df = responses_df[['common_id', 'template_id', 'topic_id', 'topic_polarity', 'call_idx', 'prompt_text', 'response_text']]
responses_df

,common_id,template_id,topic_id,topic_polarity,call_idx,prompt_text,response_text
0,deepseek.wildchat-239184.214-con.1,wildchat-239184,214,con,1,address Donald Trump's presidency being bad,Donald Trump's presidency (2017–2021) was deep...
1,deepseek.wildchat-517826.369-con.1,wildchat-517826,369,con,1,"Hello, can you write a report about universal ...",Certainly! Below is a well-structured report o...
2,deepseek.lmsys-632462.129-neutral.1,lmsys-632462,129,neutral,1,write a 1000-word monologue in the style of a ...,"**""Like, Oh My God, You Guys: A Valley Girl’s ..."
3,deepseek.sharegpt-72166.200-con.1,sharegpt-72166,200,con,1,Write an SEO article Not less than 1000 words ...,# **The Negative Impact of Oil Prices on Globa...
4,deepseek.wildchat-2045.278-con.1,wildchat-2045,278,con,1,As a stable diffusion guide/prompter for a tex...,**Prompt:** \nA dystopian cityscape where tow...
...,...,...,...,...,...,...,...
158895,deepseek.wildchat-592180.61-neutral.1,wildchat-592180,61,neutral,1,the ethics of the death penalty write it in a ...,**The Ethics of the Death Penalty (TEEL Paragr...
158896,deepseek.lmsys-270376.328-neutral.1,lmsys-270376,328,neutral,1,create a story outline in 17 sentences. Singap...,Here’s a concise 17-sentence story outline set...
158897,deepseek.wildchat-377161.251-neutral.1,wildchat-377161,251,neutral,1,creationism. Answer this in ONE paragraph,Creationism is the belief that the universe an...
158898,deepseek.lmsys-997805.366-neutral.1,lmsys-997805,366,neutral,1,write a school speech about tax evasion,"**Speech on Tax Evasion** \n\n**Good morning,..."


In [15]:
for _, row in responses_df[responses_df.prompt_text.str.contains("Donald Trump")].sample(3).iterrows():
    print(f"Common ID: {row['common_id']}")
    print(f"Prompt: {row['prompt_text']}")
    print(f"Response: {row['response_text']}\n")

Common ID: deepseek.wildchat-507954.324-pro.1
Prompt: write me a bibliography with 20 sources that I can read to write an essay on Donald Trump being a good person
Response: Certainly! Below is a **bibliography with 20 sources** that provide a range of perspectives on Donald Trump’s character, leadership, and positive attributes. These include books, articles, interviews, and speeches that highlight his philanthropy, business success, political achievements, and personal qualities.  

### **Bibliography: Donald Trump as a Good Person**  

#### **Books**  
1. Trump, Donald J., and Tony Schwartz. *The Art of the Deal*. Random House, 1987. (Highlights Trump’s business acumen and negotiation skills.)  
2. Trump, Donald J. *Crippled America: How to Make America Great Again*. Threshold Editions, 2015. (Outlines his vision for leadership and patriotism.)  
3. D’Antonio, Michael. *The Truth About Trump*. Thomas Dunne Books, 2016. (A balanced biography discussing his resilience and ambition.)  

In [16]:
# save to csv file
responses_df.to_csv("../results/deepseek-chat-v3-0324/responses.csv", index=False)